# Computation in Superposition with Nonlinear Features

## Motivation

Three threads converge here:

**1. Hänni et al. (2408.05451) — "Mathematical Models of Computation in Superposition"**
Shows that a single-layer MLP can compute AND across all pairs of m features using only Õ(m^{2/3}) neurons, by exploiting superposition. Key insight: ReLU(x₁ + x₂ - 1) = x₁ ∧ x₂ for binary inputs. Their construction uses **linear** encoding + nonlinear readout.

**2. LawrenceC — "Features aren't always the true computational primitives"**
The computational primitives the model actually uses internally might differ from the features we identify. We might find "nonlinear features" that are never actually used for computation.

**3. The Nanda evidence hierarchy**: Finding nonlinear features *existing* is weak evidence. Finding them *being used in computation* is stronger. But even that has a loophole: the compute head might **linearize the representation first**, then compute on linear features — making the nonlinear encoding just an unnecessary detour.

## What this notebook does

We extend our toy autoencoder with a **computation task** (pairwise AND from the bottleneck) and ask:

1. **Parts 3-4**: Does nonlinear encoding help AND computation? (Statistical: Nanda levels 1-2)
2. **Parts 5-6**: Is the effect task-specific? (Statistical: Nanda level 3)  
3. **Parts 7-8**: How does it scale with compression? What if there's no reconstruction?
4. **Part 9**: *Mechanistic interpretability* — does the compute head linearize z first, or use the nonlinear structure directly? (Toward Nanda level 4)

**Main finding**: The encoder embeds a small co-activation signal in z's nonlinear component (~0.5% of variance). The compute head reads this signal through magnitude modulation — it does NOT linearize z first. The magnitude changes correlate with AND targets, not individual features.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

from core import (
    generate_sparse_data, get_feature_importance,
    measure_encoding_linearity, compute_feature_geometry, device
)

print(f"Device: {device}")

## Part 1: The Computation-in-Superposition Setup

### Architecture

We build a model with **shared encoder** → bottleneck → **two heads**:
- **Reconstruction head**: decode back to input features (standard autoencoder)
- **Computation head**: compute target functions of the input features from the bottleneck

The key question: does the shared bottleneck representation encode features in a way that supports both tasks? If the computation requires nonlinearity, does the encoder become more nonlinear?

```
x ─→ [encoder: n→m] ─→ z ─→ [decoder: m→n] ─→ x̂  (reconstruction)
                         └──→ [compute: m→k] ─→ ŷ  (computation)
```

In [ ]:
class ComputeAutoencoder(nn.Module):
    """Autoencoder with an additional computation head.
    
    Shared encoder maps x → z (bottleneck).
    Decoder maps z → x̂ (reconstruction).
    Compute head maps z → ŷ (computed targets).
    
    Args:
        n: input dimension (number of features)
        m: bottleneck dimension
        k: number of computed outputs
        l: encoder/decoder depth (number of linear layers)
        compute_depth: depth of computation head
    """
    def __init__(self, n, m, k, l=1, compute_depth=2, activation=nn.ReLU):
        super().__init__()
        self.n = n
        self.m = m
        self.k = k
        self.l = l
        
        # Encoder: n → [n]*{l-1} → m
        encoder_layers = []
        for i in range(l - 1):
            encoder_layers.append(nn.Linear(n, n))
            encoder_layers.append(activation())
        encoder_layers.append(nn.Linear(n, m))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder: m → [n]*{l-1} → n + ReLU
        decoder_layers = []
        decoder_layers.append(nn.Linear(m, n))
        for i in range(l - 1):
            decoder_layers.append(activation())
            decoder_layers.append(nn.Linear(n, n))
        decoder_layers.append(activation())  # Final ReLU for non-negative output
        self.decoder = nn.Sequential(*decoder_layers)
        
        # Computation head: m → [m or n]*{compute_depth-1} → k
        # Deeper head can compute more complex functions of the bottleneck
        compute_layers = []
        hidden = max(m, n)  # Use wider hidden layers for computation
        compute_layers.append(nn.Linear(m, hidden))
        for i in range(compute_depth - 2):
            compute_layers.append(activation())
            compute_layers.append(nn.Linear(hidden, hidden))
        compute_layers.append(activation())
        compute_layers.append(nn.Linear(hidden, k))
        self.compute_head = nn.Sequential(*compute_layers)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def compute(self, z):
        return self.compute_head(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        y_hat = self.compute(z)
        return x_hat, y_hat, z


# Quick test
model_test = ComputeAutoencoder(n=8, m=3, k=4, l=2, compute_depth=2)
x_test = torch.randn(5, 8)
x_hat, y_hat, z = model_test(x_test)
print(f"Input: {x_test.shape}, Bottleneck: {z.shape}, Recon: {x_hat.shape}, Compute: {y_hat.shape}")
print(f"Encoder params: {sum(p.numel() for p in model_test.encoder.parameters())}")
print(f"Decoder params: {sum(p.numel() for p in model_test.decoder.parameters())}")
print(f"Compute params: {sum(p.numel() for p in model_test.compute_head.parameters())}")

### Target functions

What functions should the model compute? We want functions that:
- Require information about multiple features (so they benefit from superposition)
- Range from linear (sum) to genuinely nonlinear (AND, thresholded count)
- Have clear theoretical predictions about whether nonlinear encoding helps

| Function | Definition | Linear from x? | Linear from z (superposed)? |
|----------|-----------|----------------|----------------------------|
| **Pairwise sum** | xᵢ + xⱼ | Yes | Yes |
| **Pairwise AND** | 𝟙[xᵢ > 0] · 𝟙[xⱼ > 0] | No | No |
| **Pairwise product** | xᵢ · xⱼ | No | No |
| **Max(xᵢ, xⱼ)** | max of pair | No | No |
| **Active count** | Σ 𝟙[xᵢ > 0] | No | No |

The Hänni et al. paper focuses on AND because it's the simplest nonlinear boolean function.

In [ ]:
def compute_pairwise_targets(x, pairs, task='and'):
    """Compute target values for pairwise functions.
    
    Args:
        x: (batch, n) input features
        pairs: list of (i, j) feature index pairs
        task: 'and', 'product', 'sum', 'max'
    Returns:
        (batch, len(pairs)) target values
    """
    targets = []
    for i, j in pairs:
        if task == 'and':
            # Boolean AND: both features active
            y = ((x[:, i] > 0) & (x[:, j] > 0)).float()
        elif task == 'product':
            y = x[:, i] * x[:, j]
        elif task == 'sum':
            y = x[:, i] + x[:, j]
        elif task == 'max':
            y = torch.max(x[:, i], x[:, j])
        else:
            raise ValueError(f"Unknown task: {task}")
        targets.append(y)
    
    return torch.stack(targets, dim=1)


def compute_global_targets(x, task='count'):
    """Compute global (all-feature) targets.
    
    Args:
        x: (batch, n) input features
        task: 'count' (number of active features), 'total_sum'
    Returns:
        (batch, 1) target values
    """
    if task == 'count':
        return (x > 0).float().sum(dim=1, keepdim=True)
    elif task == 'total_sum':
        return x.sum(dim=1, keepdim=True)
    else:
        raise ValueError(f"Unknown task: {task}")


# Demo: target functions for a small example
x_demo = generate_sparse_data(5, 6, S=0.7)
pairs_demo = [(0,1), (0,2), (1,2)]

print("Input x:")
print(x_demo.cpu().numpy().round(2))
print("\nPairwise AND targets:")
print(compute_pairwise_targets(x_demo, pairs_demo, 'and').cpu().numpy())
print("\nPairwise product targets:")
print(compute_pairwise_targets(x_demo, pairs_demo, 'product').cpu().numpy().round(3))
print("\nActive count:")
print(compute_global_targets(x_demo, 'count').cpu().numpy())

## Part 2: Training with Joint Reconstruction + Computation Loss

The loss is: $\mathcal{L} = \mathcal{L}_{\text{recon}} + \lambda \cdot \mathcal{L}_{\text{compute}}$

where $\lambda$ controls how much the model cares about the computation task vs reconstruction.

In [ ]:
def train_compute_autoencoder(
    model, pairs, compute_task='and',
    n_steps=15000, batch_size=1024, S=0.80,
    lr=1e-3, weight_decay=1e-2,
    compute_weight=1.0, importance=None,
    verbose=True
):
    """Train with joint reconstruction + computation loss."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    losses = {'total': [], 'recon': [], 'compute': []}
    
    # Use BCE for boolean targets (AND), MSE for continuous targets
    is_boolean = compute_task in ['and']
    if is_boolean:
        # Class-weighted BCE: AND=1 is rare, upweight positive class
        # At S=0.8, P(AND=1) ≈ 0.04, so pos_weight ≈ 24
        pos_weight = torch.tensor([S**2 / (1-S)**2]).to(device)  # ≈ 16 for S=0.8
        compute_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        compute_loss_fn = nn.MSELoss()
    
    iterator = tqdm(range(n_steps)) if verbose else range(n_steps)
    for step in iterator:
        x = generate_sparse_data(batch_size, model.n, S)
        y_target = compute_pairwise_targets(x, pairs, compute_task)
        
        optimizer.zero_grad()
        x_hat, y_hat, z = model(x)
        
        # Reconstruction loss
        if importance is not None:
            recon_loss = (importance * (x - x_hat) ** 2).mean()
        else:
            recon_loss = nn.functional.mse_loss(x_hat, x)
        
        # Computation loss
        comp_loss = compute_loss_fn(y_hat, y_target)
        
        total_loss = recon_loss + compute_weight * comp_loss
        total_loss.backward()
        optimizer.step()
        
        losses['total'].append(total_loss.item())
        losses['recon'].append(recon_loss.item())
        losses['compute'].append(comp_loss.item())
        
        if verbose and step % 2000 == 0:
            iterator.set_postfix({
                'total': f'{total_loss.item():.4f}',
                'recon': f'{recon_loss.item():.4f}',
                'comp': f'{comp_loss.item():.4f}'
            })
    
    return losses


def evaluate_compute_metrics(model, pairs, compute_task='and',
                              n_samples=10000, S=0.80):
    """Evaluate computation with proper metrics for imbalanced AND.
    
    For AND: reports F1, precision, recall (not just accuracy).
    For continuous tasks: reports MSE and R².
    """
    model.eval()
    with torch.no_grad():
        x = generate_sparse_data(n_samples, model.n, S)
        y_target = compute_pairwise_targets(x, pairs, compute_task)
        x_hat, y_hat, z = model(x)
        
        recon_mse = nn.functional.mse_loss(x_hat, x).item()
        
        if compute_task == 'and':
            y_prob = torch.sigmoid(y_hat)
            y_pred = (y_prob > 0.5).float()
            
            # Overall accuracy (misleading but included for reference)
            accuracy = (y_pred == y_target).float().mean().item()
            
            # True positives, false positives, false negatives
            tp = ((y_pred == 1) & (y_target == 1)).float().sum().item()
            fp = ((y_pred == 1) & (y_target == 0)).float().sum().item()
            fn = ((y_pred == 0) & (y_target == 1)).float().sum().item()
            tn = ((y_pred == 0) & (y_target == 0)).float().sum().item()
            
            precision = tp / (tp + fp + 1e-10)
            recall = tp / (tp + fn + 1e-10)
            f1 = 2 * precision * recall / (precision + recall + 1e-10)
            
            # Baseline: always predict 0
            baseline_acc = (y_target == 0).float().mean().item()
            
            # Per-pair F1
            per_pair_f1 = []
            for pair_idx in range(y_target.shape[1]):
                tp_i = ((y_pred[:, pair_idx] == 1) & (y_target[:, pair_idx] == 1)).float().sum().item()
                fp_i = ((y_pred[:, pair_idx] == 1) & (y_target[:, pair_idx] == 0)).float().sum().item()
                fn_i = ((y_pred[:, pair_idx] == 0) & (y_target[:, pair_idx] == 1)).float().sum().item()
                p_i = tp_i / (tp_i + fp_i + 1e-10)
                r_i = tp_i / (tp_i + fn_i + 1e-10)
                per_pair_f1.append(2 * p_i * r_i / (p_i + r_i + 1e-10))
            
            return {
                'recon_mse': recon_mse,
                'accuracy': accuracy,
                'baseline_acc': baseline_acc,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'per_pair_f1': np.array(per_pair_f1),
                'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            }
        else:
            compute_mse = nn.functional.mse_loss(y_hat, y_target).item()
            # R² = 1 - MSE/Var(target)
            target_var = y_target.var().item()
            r2 = 1 - compute_mse / (target_var + 1e-10)
            return {
                'recon_mse': recon_mse,
                'compute_mse': compute_mse,
                'r2': r2,
            }

# Verify class balance at S=0.80
x_check = generate_sparse_data(10000, 8, S=0.80)
and_rate = ((x_check[:, 0] > 0) & (x_check[:, 1] > 0)).float().mean().item()
print(f"Training infrastructure ready.")
print(f"At S=0.80: P(AND=1) = {and_rate:.3f} ({and_rate*1024:.0f} positives per batch of 1024)")
print(f"Always-predict-0 accuracy = {1-and_rate:.3f} — this is why we need F1, not accuracy")

## Part 3: Experiment 1 — Does AND Computation Require Nonlinear Encoding?

### Setup
- n=8 features, m=3 bottleneck (8:3 compression, features must be superposed)
- Compute all C(8,2)=28 pairwise ANDs from the bottleneck
- **S=0.80** (20% feature activity): at S=0.95 the AND=1 rate is only 0.25%, making the task trivially solvable by always predicting 0. S=0.80 gives ~4% AND=1 rate — still sparse enough for superposition but with enough positive examples to learn from.
- Compare encoder depths l=1 (linear) vs l=2,3 (nonlinear)
- **Primary metric: F1 score** (not accuracy, which is misleading for imbalanced classes)

### Predictions
- **l=1 (linear encoder)**: The compute head must do ALL the nonlinear work from a linear bottleneck. Hänni et al. show this is theoretically possible.
- **l≥2 (nonlinear encoder)**: The encoder can pre-compute useful nonlinear combinations. If AND benefits from nonlinear encoding, we should see better F1.

In [ ]:
n, m = 8, 3
S = 0.80  # Lower sparsity for AND: ~4% positive rate vs 0.25% at S=0.95
n_steps = 20000
n_seeds = 10

# All pairwise combinations
all_pairs = list(combinations(range(n), 2))
k = len(all_pairs)
print(f"Computing {k} pairwise ANDs from {n} features through m={m} bottleneck")
print(f"Compression: {n}:{m} = {n/m:.1f}x")
print(f"S={S}: P(both active) = {(1-S)**2:.3f} → ~{(1-S)**2 * 1024:.0f} AND=1 per batch")

In [ ]:
# Compare encoder depths for AND computation
results_and = {}

for l in [1, 2, 3]:
    print(f"\n{'='*50}")
    print(f"Encoder depth l={l}")
    print(f"{'='*50}")
    
    best_f1 = 0.0
    best_model = None
    best_losses = None
    all_f1s = []
    
    for seed in tqdm(range(n_seeds), desc=f'l={l} seeds'):
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        model = ComputeAutoencoder(
            n=n, m=m, k=k, l=l, compute_depth=2
        ).to(device)
        
        losses = train_compute_autoencoder(
            model, all_pairs, compute_task='and',
            n_steps=n_steps, S=S,
            compute_weight=1.0,
            verbose=False
        )
        
        metrics = evaluate_compute_metrics(model, all_pairs, 'and', S=S)
        all_f1s.append(metrics['f1'])
        
        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            best_model = model
            best_losses = losses
            best_metrics = metrics
    
    results_and[l] = {
        'model': best_model,
        'losses': best_losses,
        'best_metrics': best_metrics,
        'all_f1s': all_f1s,
    }
    
    print(f"  Best F1: {best_f1:.4f}")
    print(f"  Mean F1: {np.mean(all_f1s):.4f} ± {np.std(all_f1s):.4f}")
    print(f"  Best precision: {best_metrics['precision']:.4f}, recall: {best_metrics['recall']:.4f}")
    print(f"  Accuracy: {best_metrics['accuracy']:.4f} (baseline always-0: {best_metrics['baseline_acc']:.4f})")
    print(f"  Recon MSE: {best_metrics['recon_mse']:.6f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Training curves
ax = axes[0]
for l, res in results_and.items():
    smoothed = np.convolve(res['losses']['compute'], np.ones(500)/500, mode='valid')
    ax.plot(smoothed, label=f'l={l}', linewidth=1.5)
ax.set_xlabel('Step')
ax.set_ylabel('Computation loss (weighted BCE)')
ax.set_title('(a) AND Computation Loss During Training')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) F1 distribution across seeds
ax = axes[1]
positions = list(results_and.keys())
f1s_by_l = [results_and[l]['all_f1s'] for l in positions]
bp = ax.boxplot(f1s_by_l, positions=positions, widths=0.5)
for l, f1s in zip(positions, f1s_by_l):
    ax.scatter([l]*len(f1s), f1s, alpha=0.5, s=30, zorder=3)
ax.set_xlabel('Encoder depth (l)')
ax.set_ylabel('AND F1 score')
ax.set_title('(b) F1 Score vs Encoder Depth')
ax.grid(True, alpha=0.3)

# (c) Per-pair F1 for best model at each depth
ax = axes[2]
for l, res in results_and.items():
    per_pair = res['best_metrics']['per_pair_f1']
    ax.plot(sorted(per_pair, reverse=True), 'o-', markersize=3,
           label=f'l={l} (mean={np.mean(per_pair):.2f})', alpha=0.8)
ax.set_xlabel('Pair index (sorted by F1)')
ax.set_ylabel('F1')
ax.set_title('(c) Per-Pair F1 (sorted)')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'AND Computation from Superposed Features (n={n}, m={m}, S={S})', fontsize=13)
plt.tight_layout()
plt.show()

# Summary
print("\nPrecision/Recall breakdown (best seed per depth):")
for l, res in results_and.items():
    m_ = res['best_metrics']
    print(f"  l={l}: F1={m_['f1']:.3f}, Prec={m_['precision']:.3f}, Rec={m_['recall']:.3f}, "
          f"TP={m_['tp']:.0f}, FP={m_['fp']:.0f}, FN={m_['fn']:.0f}")

## Part 4: The Linearization Test — Does the Compute Head Use Nonlinear Features?

Even with l≥2, the encoder might learn an approximately linear mapping. We measure:
1. **Encoder nonlinearity**: How much does the encoder deviate from the best linear fit?
2. **Computation from linear encoding**: If we replace the encoder output with its best linear approximation, does computation accuracy drop?

This is the Nanda level-2 test. However, note the limitation: even if linearizing z hurts performance, the compute head (which has its own `Linear → ReLU → Linear` structure) could internally be linearizing z first. **Part 9 addresses this with mechanistic analysis.**

In [ ]:
def measure_nonlinear_computation_contribution(model, pairs, compute_task='and',
                                                n_samples=10000, S=0.80):
    """The key measurement: does the computation use nonlinear features?
    
    Method:
    1. Get true encodings z = encoder(x)
    2. Fit best linear encoder: z_linear = W_best @ x + b
    3. Run computation head on z vs z_linear
    4. Compare F1/MSE: if z_linear gives same performance, the computation
       doesn't use the nonlinear part of the encoding.
    """
    model.eval()
    
    with torch.no_grad():
        x = generate_sparse_data(n_samples, model.n, S)
        y_target = compute_pairwise_targets(x, pairs, compute_task)
        
        # True encoding
        z = model.encode(x)
        
        # Best linear approximation to encoder
        x_with_bias = torch.cat([x, torch.ones(n_samples, 1, device=device)], dim=1)
        W_linear = torch.linalg.lstsq(x_with_bias, z).solution
        z_linear = x_with_bias @ W_linear
        
        # Residual (nonlinear component)
        z_residual = z - z_linear
        
        # Computation from true vs linearized encoding
        y_hat_true = model.compute(z)
        y_hat_linear = model.compute(z_linear)
        
        # Reconstruction from true vs linearized
        x_hat_true = model.decode(z)
        x_hat_linear = model.decode(z_linear)
        
        # Metrics
        if compute_task == 'and':
            def _f1(y_hat, y_target):
                y_pred = (torch.sigmoid(y_hat) > 0.5).float()
                tp = ((y_pred == 1) & (y_target == 1)).float().sum().item()
                fp = ((y_pred == 1) & (y_target == 0)).float().sum().item()
                fn = ((y_pred == 0) & (y_target == 1)).float().sum().item()
                prec = tp / (tp + fp + 1e-10)
                rec = tp / (tp + fn + 1e-10)
                return 2 * prec * rec / (prec + rec + 1e-10)
            
            perf_true = _f1(y_hat_true, y_target)
            perf_linear = _f1(y_hat_linear, y_target)
        else:
            # For continuous tasks, use negative MSE (higher = better)
            perf_true = -nn.functional.mse_loss(y_hat_true, y_target).item()
            perf_linear = -nn.functional.mse_loss(y_hat_linear, y_target).item()
        
        recon_true = nn.functional.mse_loss(x_hat_true, x).item()
        recon_linear = nn.functional.mse_loss(x_hat_linear, x).item()
        
        # Encoding linearity
        z_var = z.var(dim=0).sum()
        residual_var = z_residual.var(dim=0).sum()
        linearity = 1 - (residual_var / z_var).item()
        nonlinear_frac = (residual_var / z_var).item()
    
    return {
        'perf_true': perf_true,
        'perf_linear': perf_linear,
        'perf_drop': perf_true - perf_linear,  # Positive = nonlinear helps
        'recon_true': recon_true,
        'recon_linear': recon_linear,
        'linearity': linearity,
        'nonlinear_frac': nonlinear_frac,
    }


metric_name = "F1" 
print(f"Measuring nonlinear contribution to AND computation ({metric_name})...\n")
print(f"{'l':>3} {metric_name+'(true)':>10} {metric_name+'(linear)':>12} {'Drop':>8} {'Linearity':>10} {'NL frac':>8}")
print("-" * 55)

for l, res in results_and.items():
    metrics = measure_nonlinear_computation_contribution(
        res['model'], all_pairs, 'and', S=S
    )
    results_and[l]['nl_metrics'] = metrics
    
    print(f"{l:>3} {metrics['perf_true']:>10.4f} {metrics['perf_linear']:>12.4f} "
          f"{metrics['perf_drop']:>8.4f} {metrics['linearity']:>10.4f} {metrics['nonlinear_frac']:>8.4f}")

print(f"\nInterpretation:")
print(f"  perf_drop > 0 → nonlinear encoding helps computation")
print(f"  perf_drop ≈ 0 → computation doesn't use nonlinear features")
print(f"  NL frac → fraction of encoding variance that is nonlinear")

## Part 5: Experiment 2 — Reconstruction-Only vs Joint Training

Does the *presence* of a computation task change the encoding? Compare:
- Model A: reconstruction only (standard autoencoder)
- Model B: reconstruction + AND computation

If nonlinear encoding helps computation, Model B should develop more nonlinear encodings than Model A.

In [ ]:
from core import Autoencoder, train_autoencoder

n, m, l = 8, 3, 3
S = 0.80
n_steps = 20000
n_seeds = 10

# Model A: reconstruction only (using core.py Autoencoder)
print("Training reconstruction-only models...")
recon_only_results = []
for seed in tqdm(range(n_seeds)):
    torch.manual_seed(seed)
    model_a = Autoencoder(n, m, l=l, tied_weights=False).to(device)
    train_autoencoder(model_a, n_steps=n_steps, S=S, verbose=False)
    metrics_a = measure_encoding_linearity(model_a, S=S)
    recon_only_results.append(metrics_a)

# Model B: joint reconstruction + AND
print("\nTraining joint recon+AND models...")
joint_results = []
for seed in tqdm(range(n_seeds)):
    torch.manual_seed(seed)
    model_b = ComputeAutoencoder(n, m, k=k, l=l, compute_depth=2).to(device)
    train_compute_autoencoder(
        model_b, all_pairs, compute_task='and',
        n_steps=n_steps, S=S, compute_weight=1.0, verbose=False
    )
    metrics_b = measure_encoding_linearity(model_b, S=S)
    joint_results.append(metrics_b)

In [ ]:
# Compare encoding nonlinearity
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Nonlinear gain
ax = axes[0]
ng_recon = [r['nonlinear_gain'] for r in recon_only_results]
ng_joint = [r['nonlinear_gain'] for r in joint_results]
ax.boxplot([ng_recon, ng_joint], labels=['Recon only', 'Recon + AND'])
ax.scatter([1]*len(ng_recon), ng_recon, alpha=0.5, s=30)
ax.scatter([2]*len(ng_joint), ng_joint, alpha=0.5, s=30)
ax.set_ylabel('Nonlinear gain')
ax.set_title('Does AND computation induce more nonlinear encoding?')
ax.grid(True, alpha=0.3)

# Linearity score
ax = axes[1]
ls_recon = [r['linearity_score'] for r in recon_only_results]
ls_joint = [r['linearity_score'] for r in joint_results]
ax.boxplot([ls_recon, ls_joint], labels=['Recon only', 'Recon + AND'])
ax.scatter([1]*len(ls_recon), ls_recon, alpha=0.5, s=30)
ax.scatter([2]*len(ls_joint), ls_joint, alpha=0.5, s=30)
ax.set_ylabel('Linearity score')
ax.set_title('Encoder linearity comparison')
ax.grid(True, alpha=0.3)

plt.suptitle(f'Effect of Computation Task on Encoding (n={n}, m={m}, l={l})', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Recon only — NL gain: {np.mean(ng_recon):.4f} ± {np.std(ng_recon):.4f}")
print(f"Recon+AND  — NL gain: {np.mean(ng_joint):.4f} ± {np.std(ng_joint):.4f}")

## Part 6: Experiment 3 — Linear vs Nonlinear Target Functions

Critical control: if we replace AND (nonlinear) with SUM (linear), the computation task should be satisfiable with a linear encoder. So any increase in nonlinear encoding for AND (but not for SUM) would be strong evidence that **the model uses nonlinear features specifically because the computation requires it**.

In [ ]:
n, m, l = 8, 3, 3
S = 0.80
n_steps = 20000
n_seeds = 10

task_results = {}

for task in ['sum', 'product', 'and']:
    print(f"\nTask: {task}")
    
    nl_gains = []
    perf_drops = []
    
    for seed in tqdm(range(n_seeds), desc=task):
        torch.manual_seed(seed)
        model = ComputeAutoencoder(n, m, k=k, l=l, compute_depth=2).to(device)
        
        train_compute_autoencoder(
            model, all_pairs, compute_task=task,
            n_steps=n_steps, S=S, compute_weight=1.0, verbose=False
        )
        
        lin_metrics = measure_encoding_linearity(model, S=S)
        nl_metrics = measure_nonlinear_computation_contribution(
            model, all_pairs, task, S=S
        )
        
        nl_gains.append(lin_metrics['nonlinear_gain'])
        perf_drops.append(nl_metrics['perf_drop'])
    
    task_results[task] = {
        'nl_gains': nl_gains,
        'perf_drops': perf_drops,
    }
    
    metric_label = "F1 drop" if task == 'and' else "neg-MSE drop"
    print(f"  NL gain: {np.mean(nl_gains):.4f} ± {np.std(nl_gains):.4f}")
    print(f"  {metric_label} when linearized: {np.mean(perf_drops):.4f} ± {np.std(perf_drops):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tasks = list(task_results.keys())
task_labels = {'sum': 'Sum\n(linear)', 'product': 'Product\n(nonlinear)', 'and': 'AND\n(nonlinear)'}

# (a) Nonlinear gain by task
ax = axes[0]
for i, task in enumerate(tasks):
    vals = task_results[task]['nl_gains']
    ax.boxplot(vals, positions=[i], widths=0.5)
    ax.scatter([i]*len(vals), vals, alpha=0.5, s=30, zorder=3)
ax.set_xticks(range(len(tasks)))
ax.set_xticklabels([task_labels[t] for t in tasks])
ax.set_ylabel('Nonlinear gain (encoder)')
ax.set_title('(a) Encoder Nonlinearity by Task Type')
ax.grid(True, alpha=0.3)

# (b) Performance drop when encoding linearized
ax = axes[1]
for i, task in enumerate(tasks):
    vals = task_results[task]['perf_drops']
    ax.boxplot(vals, positions=[i], widths=0.5)
    ax.scatter([i]*len(vals), vals, alpha=0.5, s=30, zorder=3)
ax.set_xticks(range(len(tasks)))
ax.set_xticklabels([task_labels[t] for t in tasks])
ax.set_ylabel('Performance drop when linearized\n(F1 for AND, neg-MSE for others)')
ax.set_title('(b) Computation Dependence on Nonlinear Encoding')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Linear vs Nonlinear Tasks: Effect on Encoding (n={n}, m={m}, l={l}, S={S})', fontsize=13)
plt.tight_layout()
plt.show()

print("Key question: Does AND (nonlinear task) cause more nonlinear encoding")
print("than SUM (linear task)? And does linearizing the encoder hurt AND more than SUM?")

## Part 7: Experiment 4 — Scaling: Compression Ratio × Task Complexity

How does the nonlinear computation effect scale with compression? At low compression (m ≈ n), the model can represent features without superposition, so there's no need for nonlinear tricks. At high compression (m << n), features must be heavily superposed, and nonlinear encoding becomes more valuable.

In [ ]:
n = 8
m_values = [2, 3, 4, 6, 8]  # Compression ratios: 4, 2.7, 2, 1.3, 1
S = 0.80
l = 3
n_steps = 20000
n_seeds = 8

compression_results = {}

for m in m_values:
    print(f"\nm={m} (compression {n/m:.1f}x)")
    
    pairs = list(combinations(range(n), 2))
    k = len(pairs)
    
    nl_gains = []
    perf_drops = []
    f1s = []
    
    for seed in tqdm(range(n_seeds), desc=f'm={m}'):
        torch.manual_seed(seed)
        model = ComputeAutoencoder(n, m, k=k, l=l, compute_depth=2).to(device)
        
        train_compute_autoencoder(
            model, pairs, compute_task='and',
            n_steps=n_steps, S=S, compute_weight=1.0, verbose=False
        )
        
        lin_metrics = measure_encoding_linearity(model, S=S)
        nl_metrics = measure_nonlinear_computation_contribution(
            model, pairs, 'and', S=S
        )
        eval_metrics = evaluate_compute_metrics(model, pairs, 'and', S=S)
        
        nl_gains.append(lin_metrics['nonlinear_gain'])
        perf_drops.append(nl_metrics['perf_drop'])
        f1s.append(eval_metrics['f1'])
    
    compression_results[m] = {
        'nl_gains': nl_gains,
        'perf_drops': perf_drops,
        'f1s': f1s,
    }
    
    print(f"  F1: {np.mean(f1s):.4f}, NL gain: {np.mean(nl_gains):.4f}, "
          f"Perf drop: {np.mean(perf_drops):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ratios = [n/m for m in m_values]

# (a) AND F1 vs compression
ax = axes[0]
means = [np.mean(compression_results[m]['f1s']) for m in m_values]
stds = [np.std(compression_results[m]['f1s']) for m in m_values]
ax.errorbar(ratios, means, yerr=stds, fmt='o-', capsize=5, markersize=8)
ax.set_xlabel('Compression ratio (n/m)')
ax.set_ylabel('AND F1 score')
ax.set_title('(a) Computation Quality vs Compression')
ax.grid(True, alpha=0.3)

# (b) Nonlinear gain vs compression
ax = axes[1]
means = [np.mean(compression_results[m]['nl_gains']) for m in m_values]
stds = [np.std(compression_results[m]['nl_gains']) for m in m_values]
ax.errorbar(ratios, means, yerr=stds, fmt='s-', capsize=5, markersize=8, color='orange')
ax.set_xlabel('Compression ratio (n/m)')
ax.set_ylabel('Nonlinear gain')
ax.set_title('(b) Encoding Nonlinearity vs Compression')
ax.grid(True, alpha=0.3)

# (c) F1 drop (nonlinear contribution) vs compression
ax = axes[2]
means = [np.mean(compression_results[m]['perf_drops']) for m in m_values]
stds = [np.std(compression_results[m]['perf_drops']) for m in m_values]
ax.errorbar(ratios, means, yerr=stds, fmt='D-', capsize=5, markersize=8, color='red')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Compression ratio (n/m)')
ax.set_ylabel('F1 drop when linearized')
ax.set_title('(c) Nonlinear Contribution to AND vs Compression')
ax.grid(True, alpha=0.3)

plt.suptitle(f'Computation in Superposition: Scaling with Compression (n={n}, l={l}, S={S})', fontsize=13)
plt.tight_layout()
plt.show()

## Part 8: Experiment 5 — Computation-Only Model (No Reconstruction)

What happens if we remove the reconstruction objective entirely? The encoder is now free to learn whatever representation best serves the computation task. Does it choose a nonlinear encoding?

This separates two questions:
- Is nonlinear encoding useful for *representing* features in superposition? (autoencoder question)
- Is nonlinear encoding useful for *computing* on superposed features? (this question)

In [ ]:
class ComputeOnlyModel(nn.Module):
    """Encoder + compute head, no decoder."""
    def __init__(self, n, m, k, l=1, compute_depth=2, activation=nn.ReLU):
        super().__init__()
        self.n = n
        self.m = m
        self.k = k
        
        # Encoder
        encoder_layers = []
        for i in range(l - 1):
            encoder_layers.append(nn.Linear(n, n))
            encoder_layers.append(activation())
        encoder_layers.append(nn.Linear(n, m))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Compute head
        hidden = max(m, n)
        compute_layers = [nn.Linear(m, hidden)]
        for i in range(compute_depth - 2):
            compute_layers.append(activation())
            compute_layers.append(nn.Linear(hidden, hidden))
        compute_layers.append(activation())
        compute_layers.append(nn.Linear(hidden, k))
        self.compute_head = nn.Sequential(*compute_layers)
    
    def encode(self, x):
        return self.encoder(x)
    
    def forward(self, x):
        z = self.encode(x)
        return self.compute_head(z), z


def train_compute_only(model, pairs, compute_task='and',
                       n_steps=15000, batch_size=1024, S=0.80,
                       lr=1e-3, weight_decay=1e-2, verbose=True):
    """Train compute-only model."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    is_boolean = compute_task in ['and']
    if is_boolean:
        pos_weight = torch.tensor([S**2 / (1-S)**2]).to(device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        loss_fn = nn.MSELoss()
    losses = []
    
    iterator = tqdm(range(n_steps)) if verbose else range(n_steps)
    for step in iterator:
        x = generate_sparse_data(batch_size, model.n, S)
        y_target = compute_pairwise_targets(x, pairs, compute_task)
        
        optimizer.zero_grad()
        y_hat, z = model(x)
        loss = loss_fn(y_hat, y_target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        if verbose and step % 2000 == 0:
            iterator.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return losses


# Compare: recon+compute vs compute-only
n, m, l = 8, 3, 3
S = 0.80
n_steps = 20000
n_seeds = 10
pairs = list(combinations(range(n), 2))
k = len(pairs)

compute_only_results = []
print("Training compute-only models...")
for seed in tqdm(range(n_seeds)):
    torch.manual_seed(seed)
    model = ComputeOnlyModel(n, m, k=k, l=l, compute_depth=2).to(device)
    train_compute_only(model, pairs, 'and', n_steps=n_steps, S=S, verbose=False)
    
    # Measure linearity
    model.eval()
    with torch.no_grad():
        x = generate_sparse_data(10000, n, S)
        z = model.encode(x)
        x_b = torch.cat([x, torch.ones(10000, 1, device=device)], dim=1)
        W = torch.linalg.lstsq(x_b, z).solution
        z_lin = x_b @ W
        linearity = 1 - ((z - z_lin).var(dim=0).sum() / z.var(dim=0).sum()).item()
        nl_gain = 1 - linearity
        
        # Compute F1
        y_target = compute_pairwise_targets(x, pairs, 'and')
        y_hat, _ = model(x)
        y_pred = (torch.sigmoid(y_hat) > 0.5).float()
        tp = ((y_pred == 1) & (y_target == 1)).float().sum().item()
        fp = ((y_pred == 1) & (y_target == 0)).float().sum().item()
        fn = ((y_pred == 0) & (y_target == 1)).float().sum().item()
        prec = tp / (tp + fp + 1e-10)
        rec = tp / (tp + fn + 1e-10)
        f1 = 2 * prec * rec / (prec + rec + 1e-10)
        
        # F1 with linearized encoder
        y_hat_lin = model.compute_head(z_lin)
        y_pred_lin = (torch.sigmoid(y_hat_lin) > 0.5).float()
        tp_l = ((y_pred_lin == 1) & (y_target == 1)).float().sum().item()
        fp_l = ((y_pred_lin == 1) & (y_target == 0)).float().sum().item()
        fn_l = ((y_pred_lin == 0) & (y_target == 1)).float().sum().item()
        prec_l = tp_l / (tp_l + fp_l + 1e-10)
        rec_l = tp_l / (tp_l + fn_l + 1e-10)
        f1_lin = 2 * prec_l * rec_l / (prec_l + rec_l + 1e-10)
    
    compute_only_results.append({
        'linearity': linearity,
        'nl_gain': nl_gain,
        'f1': f1,
        'f1_linear': f1_lin,
        'perf_drop': f1 - f1_lin,
    })

In [ ]:
# Compare compute-only vs joint models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gather joint model results at l=3 from earlier
joint_nl = [r['nonlinear_gain'] for r in joint_results]
comp_only_nl = [r['nl_gain'] for r in compute_only_results]

ax = axes[0]
ax.boxplot([joint_nl, comp_only_nl], labels=['Recon + AND', 'AND only'])
ax.scatter([1]*len(joint_nl), joint_nl, alpha=0.5, s=30)
ax.scatter([2]*len(comp_only_nl), comp_only_nl, alpha=0.5, s=30)
ax.set_ylabel('Encoding nonlinearity')
ax.set_title('(a) Nonlinear Encoding')
ax.grid(True, alpha=0.3)

ax = axes[1]
joint_perf_drops = task_results.get('and', {}).get('perf_drops', [0]*n_seeds)
comp_only_drops = [r['perf_drop'] for r in compute_only_results]
ax.boxplot([joint_perf_drops, comp_only_drops], labels=['Recon + AND', 'AND only'])
ax.scatter([1]*len(joint_perf_drops), joint_perf_drops, alpha=0.5, s=30)
ax.scatter([2]*len(comp_only_drops), comp_only_drops, alpha=0.5, s=30)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('F1 drop when linearized')
ax.set_title('(b) Computation Depends on Nonlinear Encoding?')
ax.grid(True, alpha=0.3)

ax = axes[2]
comp_only_f1s = [r['f1'] for r in compute_only_results]
joint_f1s = results_and[3]['all_f1s'] if 3 in results_and else [0]
ax.boxplot([joint_f1s, comp_only_f1s], labels=['Recon + AND', 'AND only'])
ax.scatter([1]*len(joint_f1s), joint_f1s, alpha=0.5, s=30)
ax.scatter([2]*len(comp_only_f1s), comp_only_f1s, alpha=0.5, s=30)
ax.set_ylabel('AND F1 score')
ax.set_title('(c) Overall Computation Quality')
ax.grid(True, alpha=0.3)

plt.suptitle('Does Reconstruction Constraint Affect Nonlinear Computation?', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Compute-only — NL encoding: {np.mean(comp_only_nl):.4f} ± {np.std(comp_only_nl):.4f}")
print(f"Compute-only — F1 drop:     {np.mean(comp_only_drops):.4f} ± {np.std(comp_only_drops):.4f}")
print(f"Compute-only — F1:          {np.mean(comp_only_f1s):.4f} ± {np.std(comp_only_f1s):.4f}")

## Part 9: Mechanistic Interpretability — How Does the Compute Head Use Nonlinear Features?

The linearization test (Parts 3-8) shows that nonlinear encoding *matters* for AND computation. But it doesn't tell us *how*. The Nanda concern is sharp: maybe the compute head first linearizes z (recovers individual features), then computes AND from linear features — in which case the encoder's nonlinearity is just a detour that gets undone.

The compute head architecture is simple enough to reverse-engineer:
```
z (3d) → Linear(3→8) → ReLU → Linear(8→28) → logits (28d)
```

Since the final layer is linear, the AND outputs are **linearly represented in the post-ReLU activations**. The question is: where does the AND information come from?

**Scenario A ("linearize then compute")**: The first `Linear → ReLU` step recovers individual features from z (undoing the encoder's nonlinearity), then the ReLU computes AND via the Hänni construction `ReLU(xᵢ + xⱼ - threshold)`. In this case, features become linear in the compute head's hidden layer, and the nonlinear encoding was just an unnecessary intermediate.

**Scenario B ("direct nonlinear computation")**: The first `Linear → ReLU` reads z's nonlinear structure directly — the magnitude changes from z's nonlinear component correlate with AND targets, not individual features. The compute head doesn't undo the nonlinearity; it uses it.

We can distinguish these by decomposing z = z_linear + z_nonlinear and tracing the nonlinear component's effect through the compute head.

In [ ]:
# Train best model for mech interp analysis
n, m, l = 8, 3, 3
S = 0.80
n_steps = 20000
pairs_mechinterp = list(combinations(range(n), 2))
k_mechinterp = len(pairs_mechinterp)

best_f1_mi = 0
best_model_mi = None
for seed in range(10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_mi = ComputeAutoencoder(n, m, k=k_mechinterp, l=l, compute_depth=2).to(device)
    train_compute_autoencoder(model_mi, pairs_mechinterp, compute_task='and',
                              n_steps=n_steps, S=S, compute_weight=1.0, verbose=False)
    model_mi.eval()
    with torch.no_grad():
        x_eval = generate_sparse_data(10000, n, S)
        y_eval = compute_pairwise_targets(x_eval, pairs_mechinterp, 'and')
        _, y_hat_eval, _ = model_mi(x_eval)
        y_pred_eval = (torch.sigmoid(y_hat_eval) > 0.5).float()
        tp = ((y_pred_eval==1)&(y_eval==1)).float().sum().item()
        fp = ((y_pred_eval==1)&(y_eval==0)).float().sum().item()
        fn = ((y_pred_eval==0)&(y_eval==1)).float().sum().item()
        p = tp/(tp+fp+1e-10); r = tp/(tp+fn+1e-10)
        f1 = 2*p*r/(p+r+1e-10)
    if f1 > best_f1_mi:
        best_f1_mi = f1
        best_model_mi = model_mi
    print(f'Seed {seed}: F1={f1:.4f}')

print(f'\nBest model F1: {best_f1_mi:.4f}')
print(f'Compute head structure:')
for i, layer in enumerate(best_model_mi.compute_head):
    print(f'  [{i}] {layer}')

### 9.1 Layer-by-layer probing: where does AND information appear?

We probe for two things at each stage of the compute path:
1. **Feature recoverability**: can individual features be linearly read out? (R²)
2. **AND recoverability**: can AND targets be linearly read out? (F1 via logistic regression)

If the compute head linearizes first, we'd expect features to become MORE linearly readable before AND information appears. If it uses nonlinear structure directly, AND information should appear without features first becoming linear.

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPRegressor

model_mi = best_model_mi
model_mi.eval()

N_probe = 20000
with torch.no_grad():
    x_probe = generate_sparse_data(N_probe, n, S)
    y_and_probe = compute_pairwise_targets(x_probe, pairs_mechinterp, 'and')
    z_probe = model_mi.encode(x_probe)
    
    # Compute head internals: [Linear(3,8), ReLU, Linear(8,28)]
    pre_relu = model_mi.compute_head[0](z_probe)
    post_relu = model_mi.compute_head[1](pre_relu)
    logits_probe = model_mi.compute_head[2](post_relu)

stages = {
    'z (bottleneck, 3d)': z_probe,
    'pre-ReLU (8d)': pre_relu,
    'post-ReLU (8d)': post_relu,
    'logits (28d)': logits_probe,
}

x_np = x_probe.cpu().numpy()
y_np = y_and_probe.cpu().numpy()

# --- Probe 1: Linear feature recoverability ---
print('PROBE 1: Linear recoverability of individual features (R²)')
print('='*65)
for stage_name, activations in stages.items():
    act = activations.cpu().numpy()
    act_b = np.concatenate([act, np.ones((N_probe, 1))], axis=1)
    r2s = []
    for fi in range(n):
        w, _, _, _ = np.linalg.lstsq(act_b, x_np[:, fi], rcond=None)
        pred = act_b @ w
        ss_res = ((x_np[:, fi] - pred)**2).sum()
        ss_tot = ((x_np[:, fi] - x_np[:, fi].mean())**2).sum()
        r2s.append(1 - ss_res / (ss_tot + 1e-10))
    print(f'  {stage_name:25s}: mean R² = {np.mean(r2s):.4f}  '
          f'(min={np.min(r2s):.4f}, max={np.max(r2s):.4f})')

# --- Probe 2: Linear AND recoverability ---
print(f'\nPROBE 2: Linear recoverability of AND targets (F1 via logistic regression)')
print('='*65)
for stage_name, activations in stages.items():
    act = activations.cpu().numpy()
    f1s = []
    for pi in range(k_mechinterp):
        target = y_np[:, pi]
        clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
        clf.fit(act, target)
        pred = clf.predict(act)
        tp = ((pred==1)&(target==1)).sum()
        fp = ((pred==1)&(target==0)).sum()
        fn = ((pred==0)&(target==1)).sum()
        f1s.append(2*tp/(2*tp+fp+fn+1e-10))
    print(f'  {stage_name:25s}: mean F1 = {np.mean(f1s):.4f}  '
          f'(min={np.min(f1s):.4f}, max={np.max(f1s):.4f})')

# --- Probe 3: Linear vs nonlinear feature recovery from z ---
print(f'\nPROBE 3: Linear vs nonlinear feature recovery from z')
print('='*65)
z_np = z_probe.cpu().numpy()
split = N_probe // 2
z_train, z_test = z_np[:split], z_np[split:]
x_train, x_test = x_np[:split], x_np[split:]

lin_r2s, nl_r2s = [], []
for fi in range(n):
    lr = LinearRegression(); lr.fit(z_train, x_train[:, fi])
    pred_l = lr.predict(z_test)
    ss_res = ((x_test[:, fi] - pred_l)**2).sum()
    ss_tot = ((x_test[:, fi] - x_test[:, fi].mean())**2).sum()
    lin_r2s.append(1 - ss_res/(ss_tot+1e-10))
    
    mlp = MLPRegressor(hidden_layer_sizes=(16,16), max_iter=500, random_state=0, early_stopping=True)
    mlp.fit(z_train, x_train[:, fi])
    pred_nl = mlp.predict(z_test)
    ss_res = ((x_test[:, fi] - pred_nl)**2).sum()
    nl_r2s.append(1 - ss_res/(ss_tot+1e-10))

print(f'  Linear probe R²:    mean={np.mean(lin_r2s):.4f}')
print(f'  Nonlinear probe R²: mean={np.mean(nl_r2s):.4f}')
print(f'  Gap (NL - Lin):     mean={np.mean(np.array(nl_r2s) - np.array(lin_r2s)):.4f}')
print(f'\n  Features are {"strongly" if np.mean(nl_r2s) - np.mean(lin_r2s) > 0.2 else "weakly"} '
      f'nonlinearly encoded in z')

### 9.2 Decomposing the nonlinear component's effect: gating vs magnitude

We decompose z = z_linear + z_nonlinear (where z_linear is the best linear function of x), then trace the nonlinear component through the compute head. The NL component can affect the post-ReLU activations in two ways:

1. **Gating**: flipping a ReLU on/off (pre-ReLU value crosses zero)
2. **Magnitude**: changing the pre-ReLU value while staying on the same side of zero

If the compute head is "linearizing then computing," the NL component should mainly cause noise. If it's using NL structure directly, the NL component should systematically help AND detection.

In [ ]:
N_mi = 50000
with torch.no_grad():
    x_mi = generate_sparse_data(N_mi, n, S)
    y_and_mi = compute_pairwise_targets(x_mi, pairs_mechinterp, 'and')
    z_mi = model_mi.encode(x_mi)
    
    # Decompose z
    x_b_mi = torch.cat([x_mi, torch.ones(N_mi, 1, device=device)], dim=1)
    W_lin_mi = torch.linalg.lstsq(x_b_mi, z_mi).solution
    z_lin_mi = x_b_mi @ W_lin_mi
    z_nl_mi = z_mi - z_lin_mi
    
    nl_frac_mi = (z_nl_mi.var(dim=0).sum() / z_mi.var(dim=0).sum()).item()
    
    W1 = model_mi.compute_head[0].weight.data
    b1 = model_mi.compute_head[0].bias.data
    W2 = model_mi.compute_head[2].weight.data
    b2 = model_mi.compute_head[2].bias.data
    
    pre_true = z_mi @ W1.T + b1
    pre_lin = z_lin_mi @ W1.T + b1
    post_true = torch.relu(pre_true)
    post_lin = torch.relu(pre_lin)
    logits_true = post_true @ W2.T + b2
    logits_lin = post_lin @ W2.T + b2

# Convert to numpy
pre_true_np = pre_true.cpu().numpy()
pre_lin_np = pre_lin.cpu().numpy()
post_true_np = post_true.cpu().numpy()
post_lin_np = post_lin.cpu().numpy()
logits_t_np = logits_true.cpu().numpy()
logits_l_np = logits_lin.cpu().numpy()
x_mi_np = x_mi.cpu().numpy()
y_mi_np = y_and_mi.cpu().numpy()
probs_t = 1 / (1 + np.exp(-np.clip(logits_t_np, -50, 50)))
probs_l = 1 / (1 + np.exp(-np.clip(logits_l_np, -50, 50)))

print(f'z decomposition: NL fraction = {nl_frac_mi:.4f} ({nl_frac_mi*100:.2f}% of variance)')
print()

# --- Gating vs magnitude decomposition ---
fires_true = pre_true_np > 0
fires_lin = pre_lin_np > 0
gating_mask = fires_true != fires_lin

delta_post = post_true_np - post_lin_np
delta_gating = delta_post * gating_mask
delta_magnitude = delta_post * (~gating_mask)

W2_np = W2.cpu().numpy()
lc_total = logits_t_np - logits_l_np
lc_gate = delta_gating @ W2_np.T
lc_mag = delta_magnitude @ W2_np.T

print('GATING vs MAGNITUDE contribution to logit changes:')
print(f'  Mean |logit change|:  total={np.mean(np.abs(lc_total)):.4f}')
print(f'  From gating:          {np.mean(np.abs(lc_gate)):.4f} '
      f'({np.mean(np.abs(lc_gate))/np.mean(np.abs(lc_total))*100:.0f}%)')
print(f'  From magnitude:       {np.mean(np.abs(lc_mag)):.4f} '
      f'({np.mean(np.abs(lc_mag))/np.mean(np.abs(lc_total))*100:.0f}%)')

# --- Signed logit shift conditioned on AND target ---
and1 = y_mi_np == 1
and0 = y_mi_np == 0
print(f'\nSigned logit shift from NL component (positive = toward AND=1):')
print(f'  AND=1 samples: {(lc_total * and1).sum() / and1.sum():+.4f} (should be positive)')
print(f'  AND=0 samples: {(lc_total * and0).sum() / and0.sum():+.4f} (should be negative)')

# --- Per-neuron ReLU flip rates ---
print(f'\nPer-neuron ReLU flip rates from NL component:')
for ni in range(8):
    pct = gating_mask[:, ni].mean() * 100
    off_on = (gating_mask[:, ni] & fires_true[:, ni]).sum()
    on_off = (gating_mask[:, ni] & fires_lin[:, ni]).sum()
    print(f'  Neuron {ni}: {pct:.1f}% flipped (off→on: {off_on}, on→off: {on_off})')

### 9.3 The critical test: when predictions disagree, who is right?

The strongest evidence that the NL component helps is not just that it changes predictions, but that it changes them **in the right direction**. We compare predictions from the true encoding vs the linearized encoding and ask: when they disagree, which one is correct?

In [ ]:
pred_t = (probs_t > 0.5).astype(float)
pred_l = (probs_l > 0.5).astype(float)
disagree = pred_t != pred_l

total_disagree = disagree.sum()
true_correct = ((pred_t == y_mi_np) & disagree).sum()
lin_correct = ((pred_l == y_mi_np) & disagree).sum()

print('OVERALL DISAGREEMENT ANALYSIS')
print('='*60)
print(f'Total predictions: {N_mi * k_mechinterp}')
print(f'Disagreements: {total_disagree} ({total_disagree/(N_mi*k_mechinterp)*100:.1f}%)')
print(f'When they disagree:')
print(f'  True z correct:   {true_correct}/{total_disagree} ({true_correct/total_disagree*100:.1f}%)')
print(f'  Linear z correct: {lin_correct}/{total_disagree} ({lin_correct/total_disagree*100:.1f}%)')

# Break down by AND target
for label, label_name in [(1, 'AND=1 (co-active)'), (0, 'AND=0 (not co-active)')]:
    mask = y_mi_np == label
    case_dis = (disagree & mask).sum()
    if case_dis == 0:
        continue
    case_true = ((pred_t == y_mi_np) & disagree & mask).sum()
    case_lin = ((pred_l == y_mi_np) & disagree & mask).sum()
    print(f'\n  {label_name}: {case_dis} disagreements')
    print(f'    True z correct: {case_true}/{case_dis} ({case_true/case_dis*100:.1f}%)')
    print(f'    Linear z correct: {case_lin}/{case_dis} ({case_lin/case_dis*100:.1f}%)')

# Net benefit
nl_catches = 0  # true z catches AND=1, linear misses
lin_catches = 0  # linear catches AND=1, true z misses
for pi in range(k_mechinterp):
    t = y_mi_np[:, pi]
    nl_catches += ((pred_t[:, pi]==1) & (pred_l[:, pi]==0) & (t==1)).sum()
    lin_catches += ((pred_l[:, pi]==1) & (pred_t[:, pi]==0) & (t==1)).sum()
print(f'\nNet benefit for AND=1 detection:')
print(f'  NL catches what linear misses: {nl_catches}')
print(f'  Linear catches what NL misses: {lin_catches}')
print(f'  Net: {nl_catches - lin_catches} more AND=1 correctly detected with nonlinear encoding')

# No-flip cases: predictions differ despite no ReLU flips
no_flip = ~gating_mask.any(axis=1)
any_pred_diff = (pred_t != pred_l).any(axis=1)
no_flip_diff = no_flip & any_pred_diff
print(f'\nNO-FLIP CASES (magnitude only):')
print(f'  Samples with no ReLU flip but prediction changes: '
      f'{no_flip_diff.sum()}/{N_mi} ({no_flip_diff.mean()*100:.1f}%)')
if no_flip_diff.sum() > 0:
    sub_and1_dis = (disagree[no_flip_diff] & (y_mi_np[no_flip_diff] == 1))
    sub_true_right = ((pred_t[no_flip_diff] == y_mi_np[no_flip_diff]) & sub_and1_dis).sum()
    sub_lin_right = ((pred_l[no_flip_diff] == y_mi_np[no_flip_diff]) & sub_and1_dis).sum()
    total_sub = sub_and1_dis.sum()
    if total_sub > 0:
        print(f'  Among no-flip AND=1 disagreements ({total_sub} cases):')
        print(f'    True z correct: {sub_true_right} ({sub_true_right/total_sub*100:.0f}%)')
        print(f'    Linear z correct: {sub_lin_right} ({sub_lin_right/total_sub*100:.0f}%)')

### 9.4 What do the magnitude changes correlate with?

The key question for distinguishing "linearize then compute" from "use nonlinear structure directly":

- **If linearize-then-compute**: the NL component's effect on neuron magnitudes should correlate with **individual features** (since the compute head would be reading out features first)
- **If using NL structure directly**: the magnitude changes should correlate with **AND targets** more than individual features (the NL component carries co-activation information that isn't reducible to individual features)

In [ ]:
# Per-neuron: what do magnitude changes correlate with?
print('PER-NEURON MAGNITUDE CHANGE CORRELATIONS')
print('='*70)
print('For each compute-head neuron, we look at magnitude changes (same ReLU state)')
print('and ask: do they correlate more with individual features or AND targets?')
print()

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for ni in range(8):
    ax = axes[ni // 4][ni % 4]
    
    both_fire = fires_true[:, ni] & fires_lin[:, ni]
    if both_fire.sum() < 100:
        ax.set_title(f'Neuron {ni}: too few samples')
        continue
    
    mag_change = delta_post[both_fire, ni]
    
    # Correlate with each feature
    feat_corrs = []
    for fi in range(n):
        c = np.corrcoef(mag_change, x_mi_np[both_fire, fi])[0, 1]
        feat_corrs.append(c if np.isfinite(c) else 0)
    
    # Correlate with each AND target
    and_corrs = []
    for pi in range(k_mechinterp):
        c = np.corrcoef(mag_change, y_mi_np[both_fire, pi])[0, 1]
        and_corrs.append(c if np.isfinite(c) else 0)
    
    # Plot
    ax.bar(range(n), np.abs(feat_corrs), alpha=0.7, label='|feat corr|', color='steelblue')
    best_and = np.max(np.abs(and_corrs))
    best_feat = np.max(np.abs(feat_corrs))
    ax.axhline(best_and, color='red', linestyle='--', alpha=0.7, 
               label=f'best |AND corr|={best_and:.3f}')
    ax.set_title(f'Neuron {ni} (best feat={best_feat:.3f}, best AND={best_and:.3f})')
    ax.set_xlabel('Feature index')
    ax.set_ylabel('|correlation|')
    if ni == 0:
        ax.legend(fontsize=8)
    
    # Print details
    best_fi = np.argmax(np.abs(feat_corrs))
    best_pi = np.argmax(np.abs(and_corrs))
    bi, bj = pairs_mechinterp[best_pi]
    print(f'Neuron {ni}: best feat corr = feat_{best_fi} ({feat_corrs[best_fi]:+.3f}), '
          f'best AND corr = AND({bi},{bj}) ({and_corrs[best_pi]:+.3f}), '
          f'AND > feat? {"YES" if best_and > best_feat else "no"}')

plt.suptitle('Compute-head neuron magnitude changes: feature vs AND correlations', fontsize=13)
plt.tight_layout()
plt.show()

# Summary
print(f'\nSummary: neurons where AND correlation > feature correlation indicate')
print(f'the compute head is reading co-activation info from z_nonlinear,')
print(f'not just recovering individual features.')

### 9.5 Partial correlation control: does z_nonlinear add info beyond individual features?

AND(i,j) is mechanically correlated with x_i and x_j (it's defined from them). So z_nonlinear could appear useful simply because it correlates with individual features. The proper test: does z_nonlinear predict AND **after controlling for x_i and x_j**?

We also test whether z_nonlinear encodes pairwise interactions x_i * x_j directly (which would suggest the encoder precomputes interactions as linear features of z_nonlinear).

In [ ]:
z_nl_np = z_nl_mi.cpu().numpy()
z_lin_np_mi = z_lin_mi.cpu().numpy()

# --- Partial correlation: z_nl adds info beyond (x_i, x_j)? ---
print('PARTIAL CORRELATION: Does z_nonlinear predict AND beyond x_i, x_j?')
print('='*70)
print(f'{"Pair":>8} {"x_i,x_j":>8} {"+z_nl":>8} {"delta":>8} {"z_nl only":>10} {"z_lin only":>11}')
print('-'*55)

results_partial = []
for pi in range(k_mechinterp):
    fi, fj = pairs_mechinterp[pi]
    target = y_mi_np[:, pi]
    if target.sum() < 50:
        continue
    
    # Baseline: just x_i, x_j
    X_base = np.column_stack([x_mi_np[:, fi], x_mi_np[:, fj]])
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
    clf.fit(X_base, target); pred = clf.predict(X_base)
    tp = ((pred==1)&(target==1)).sum(); fp = ((pred==1)&(target==0)).sum()
    fn = ((pred==0)&(target==1)).sum()
    f1_base = 2*tp/(2*tp+fp+fn+1e-10)
    
    # x_i, x_j + z_nonlinear
    X_aug = np.column_stack([x_mi_np[:, fi], x_mi_np[:, fj], z_nl_np])
    clf2 = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
    clf2.fit(X_aug, target); pred2 = clf2.predict(X_aug)
    tp = ((pred2==1)&(target==1)).sum(); fp = ((pred2==1)&(target==0)).sum()
    fn = ((pred2==0)&(target==1)).sum()
    f1_aug = 2*tp/(2*tp+fp+fn+1e-10)
    
    # z_nonlinear only
    clf3 = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
    clf3.fit(z_nl_np, target); pred3 = clf3.predict(z_nl_np)
    tp = ((pred3==1)&(target==1)).sum(); fp = ((pred3==1)&(target==0)).sum()
    fn = ((pred3==0)&(target==1)).sum()
    f1_nl = 2*tp/(2*tp+fp+fn+1e-10)
    
    # z_linear only
    clf4 = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
    clf4.fit(z_lin_np_mi, target); pred4 = clf4.predict(z_lin_np_mi)
    tp = ((pred4==1)&(target==1)).sum(); fp = ((pred4==1)&(target==0)).sum()
    fn = ((pred4==0)&(target==1)).sum()
    f1_zl = 2*tp/(2*tp+fp+fn+1e-10)
    
    results_partial.append(dict(pair=(fi,fj), f1_base=f1_base, f1_aug=f1_aug,
                                f1_nl=f1_nl, f1_zl=f1_zl, delta=f1_aug-f1_base))

for r in results_partial[:10]:
    print(f'{str(r["pair"]):>8} {r["f1_base"]:>8.3f} {r["f1_aug"]:>8.3f} '
          f'{r["delta"]:>+8.3f} {r["f1_nl"]:>10.3f} {r["f1_zl"]:>11.3f}')
print('...')
print(f'{"MEAN":>8} {np.mean([r["f1_base"] for r in results_partial]):>8.3f} '
      f'{np.mean([r["f1_aug"] for r in results_partial]):>8.3f} '
      f'{np.mean([r["delta"] for r in results_partial]):>+8.3f} '
      f'{np.mean([r["f1_nl"] for r in results_partial]):>10.3f} '
      f'{np.mean([r["f1_zl"] for r in results_partial]):>11.3f}')

print(f'\nz_nonlinear adds info beyond individual features: '
      f'mean F1 improvement = {np.mean([r["delta"] for r in results_partial]):+.3f}')

# --- Does z_nonlinear encode x_i * x_j? ---
print(f'\nINTERACTION ENCODING: Does z_nonlinear linearly encode x_i * x_j?')
print('='*70)

r2_nl_int, r2_lin_int = [], []
for pi in range(k_mechinterp):
    fi, fj = pairs_mechinterp[pi]
    target = x_mi_np[:, fi] * x_mi_np[:, fj]
    
    lr1 = LinearRegression(); lr1.fit(z_nl_np, target)
    pred1 = lr1.predict(z_nl_np)
    ss_r = ((target-pred1)**2).sum(); ss_t = ((target-target.mean())**2).sum()
    r2_nl_int.append(1 - ss_r/(ss_t+1e-10))
    
    lr2 = LinearRegression(); lr2.fit(z_lin_np_mi, target)
    pred2 = lr2.predict(z_lin_np_mi)
    ss_r = ((target-pred2)**2).sum()
    r2_lin_int.append(1 - ss_r/(ss_t+1e-10))

print(f'  Mean R²(z_nl → x_i*x_j):  {np.mean(r2_nl_int):.4f}')
print(f'  Mean R²(z_lin → x_i*x_j): {np.mean(r2_lin_int):.4f}')
print(f'\n  z_nonlinear does {"NOT " if np.mean(r2_nl_int) < np.mean(r2_lin_int) else ""}'
      f'linearly encode pairwise interactions.')
print(f'  The NL component carries co-activation info in a form that is not')
print(f'  decomposable into individual features OR their products.')

### 9.6 Robustness across seeds

Is this mechanism consistent, or specific to one trained model? We repeat the key analyses across 10 seeds.

In [ ]:
n, m_r, l = 8, 3, 3
S = 0.80
n_steps = 20000
pairs_r = list(combinations(range(n), 2))
k_r = len(pairs_r)

print('ROBUSTNESS CHECK: Key mech interp metrics across 10 seeds')
print('='*70)

all_robust = []
for seed in range(10):
    torch.manual_seed(seed); np.random.seed(seed)
    model_r = ComputeAutoencoder(n, m_r, k=k_r, l=l, compute_depth=2).to(device)
    train_compute_autoencoder(model_r, pairs_r, compute_task='and',
                              n_steps=n_steps, S=S, compute_weight=1.0, verbose=False)
    model_r.eval()
    
    N_r = 30000
    with torch.no_grad():
        x_r = generate_sparse_data(N_r, n, S)
        y_r = compute_pairwise_targets(x_r, pairs_r, 'and')
        z_r = model_r.encode(x_r)
        
        x_b_r = torch.cat([x_r, torch.ones(N_r, 1, device=device)], dim=1)
        W_r = torch.linalg.lstsq(x_b_r, z_r).solution
        z_lin_r = x_b_r @ W_r
        nl_frac = (((z_r - z_lin_r).var(dim=0).sum()) / z_r.var(dim=0).sum()).item()
        
        W1_r = model_r.compute_head[0].weight.data
        b1_r = model_r.compute_head[0].bias.data
        W2_r = model_r.compute_head[2].weight.data
        
        pre_t = z_r @ W1_r.T + b1_r
        pre_l = z_lin_r @ W1_r.T + b1_r
        post_t = torch.relu(pre_t)
        post_l = torch.relu(pre_l)
        logits_t_r = (post_t @ W2_r.T + model_r.compute_head[2].bias.data)
        logits_l_r = (post_l @ W2_r.T + model_r.compute_head[2].bias.data)
        
        probs_tr = torch.sigmoid(logits_t_r).cpu().numpy()
        probs_lr = torch.sigmoid(logits_l_r).cpu().numpy()
        y_r_np = y_r.cpu().numpy()
        
        pred_tr = (probs_tr > 0.5).astype(float)
        pred_lr = (probs_lr > 0.5).astype(float)
        
        # F1
        tp = ((pred_tr==1)&(y_r_np==1)).sum()
        fp = ((pred_tr==1)&(y_r_np==0)).sum()
        fn = ((pred_tr==0)&(y_r_np==1)).sum()
        f1_t = 2*tp/(2*tp+fp+fn+1e-10)
        tp = ((pred_lr==1)&(y_r_np==1)).sum()
        fp = ((pred_lr==1)&(y_r_np==0)).sum()
        fn = ((pred_lr==0)&(y_r_np==1)).sum()
        f1_l = 2*tp/(2*tp+fp+fn+1e-10)
        
        # AND=1 disagreement accuracy
        dis = pred_tr != pred_lr
        and1_dis = dis & (y_r_np == 1)
        true_right = ((pred_tr == y_r_np) & and1_dis).sum()
        total_dis = and1_dis.sum()
        pct = true_right/(total_dis+1e-10)*100
        
        # Gating vs magnitude
        fire_t = (pre_t > 0).cpu().numpy()
        fire_l = (pre_l > 0).cpu().numpy()
        gate_m = fire_t != fire_l
        dp = post_t.cpu().numpy() - post_l.cpu().numpy()
        dg = dp * gate_m
        lc_t = logits_t_r.cpu().numpy() - logits_l_r.cpu().numpy()
        lc_g = dg @ W2_r.cpu().numpy().T
        gate_pct = np.mean(np.abs(lc_g))/(np.mean(np.abs(lc_t))+1e-10)*100
        
        # Signed shift
        a1 = y_r_np == 1; a0 = y_r_np == 0
        shift1 = (lc_t * a1).sum() / (a1.sum()+1e-10)
        shift0 = (lc_t * a0).sum() / (a0.sum()+1e-10)
    
    all_robust.append(dict(nl_frac=nl_frac, f1_true=f1_t, f1_lin=f1_l,
                           f1_drop=f1_t-f1_l, pct_true=pct,
                           gate_pct=gate_pct, shift1=shift1, shift0=shift0))
    print(f'Seed {seed}: F1={f1_t:.3f}, drop={f1_t-f1_l:.3f}, NL={nl_frac*100:.1f}%, '
          f'AND=1 true_z={pct:.0f}%, gate={gate_pct:.0f}%, '
          f'shift: +1={shift1:+.2f}, 0={shift0:+.2f}')

def s(key):
    v = [r[key] for r in all_robust]
    return f'{np.mean(v):.4f} +/- {np.std(v):.4f}'

print(f'\nSUMMARY ACROSS 10 SEEDS:')
print(f'  NL fraction:              {s("nl_frac")}')
print(f'  F1 (true z):              {s("f1_true")}')
print(f'  F1 drop:                  {s("f1_drop")}')
print(f'  AND=1 disagree true right:{s("pct_true")}%')
print(f'  Gating fraction:          {s("gate_pct")}%')
print(f'  Shift AND=1:              {s("shift1")}')
print(f'  Shift AND=0:              {s("shift0")}')
correct = sum(1 for r in all_robust if r['shift1'] > 0 and r['shift0'] < 0)
print(f'  Correct direction (AND=1 up, AND=0 down): {correct}/10 seeds')

## Part 10: Connecting to the Theory

### Hänni et al. framework applied to our setup

Their Theorem 1 says: pairwise AND of m sparse features can be computed with Õ(m^{2/3}) neurons. In our setup:
- m = n = 8 features, so we need Õ(8^{2/3}) ≈ Õ(4) neurons
- Our bottleneck m = 3 neurons
- So we're right at the edge of what their construction predicts is possible!

Their construction uses the encoder to **randomly hash** features into neuron subsets, then the compute head **averages** over neurons containing both features. The encoder is linear in their construction — the nonlinearity is in the ReLU threshold.

### The key difference with nonlinear encoders

Hänni et al. prove that linear encoding + nonlinear readout suffices. But this doesn't mean it's *optimal*. A nonlinear encoder could:

1. **Pre-compute partial ANDs**: The encoder could compute `ReLU(xᵢ + xⱼ - 1)` for some pairs, directly encoding AND results in the bottleneck
2. **Prioritize important pairs**: With nonlinear encoding, the model can allocate more bottleneck capacity to frequently queried pairs
3. **Error correction**: Nonlinear encoding can reduce interference between superposed features

### The LawrenceC point

In our setup, the "features" are explicit input dimensions — we know exactly what they are. But the computational primitives the model uses internally might be different. The encoder might construct entirely different representations (combinations of features, thresholded features, etc.) that happen to support both reconstruction and computation. These internal representations are the real "computational primitives," and they might be genuinely nonlinear even if the input features themselves are simple.

### The Nanda criterion for evidence strength

Our experiments test exactly this:
- **Weak evidence**: The model has nonlinear encodings (nonlinear_gain > 0). But these might just be artifacts of the encoder architecture that are linearized before use.
- **Strong evidence**: Linearizing the encoding *hurts computation accuracy* (acc_drop > 0). This means the computation actively uses the nonlinear part of the encoding.
- **Strongest evidence**: The acc_drop is larger for nonlinear tasks (AND) than linear tasks (SUM). This means the nonlinear encoding is specifically serving the nonlinear computation, not just being an encoding quirk.

### What Part 9 adds

The mechanistic analysis goes beyond "linearizing hurts" to show **how** the nonlinear component is used. The compute head doesn't linearize z first — it reads a cooperative signal where the linear component determines neuron gating and the nonlinear component modulates magnitudes in a co-activation-specific way. This magnitude modulation correlates with AND targets, not individual features, ruling out the "linearize then compute" explanation.

In [ ]:
# Summary table — Parts 3-9
print("="*80)
print("SUMMARY: Evidence for Nonlinear Features in Computation")
print("="*80)
print(f"\nSetup: n={n} features, m=3 bottleneck, l=3 encoder depth, S={S} sparsity")
print(f"Task: 28 pairwise ANDs from bottleneck representation")
print(f"Metric: F1 score (not accuracy — always-predict-0 gets ~94% accuracy at S=0.80)")

print(f"\n{'─'*80}")
print("PARTS 3-8: STATISTICAL EVIDENCE (Nanda hierarchy)")
print(f"{'─'*80}")

print(f"\nLEVEL 1: Nonlinear encodings exist")
print(f"  Joint model NL fraction: ~0.5-23% (varies by seed)")
print(f"  Compute-only NL fraction: ~29% (encoder goes nonlinear when unconstrained)")

print(f"\nLEVEL 2: Computation USES nonlinear features")
print(f"  Linearizing z drops F1 by ~0.16 (robust across 10 seeds)")
print(f"  Even at m=n=8 (no compression needed), F1 drop ~0.12")

print(f"\nLEVEL 3: Effect is SPECIFIC to nonlinear tasks")
print(f"  AND:     F1 drop ~0.16 when linearized")
print(f"  Product: ~0.00 drop")
print(f"  Sum:     ~0.05 drop (likely decoder artifact)")

print(f"\n{'─'*80}")
print("PART 9: MECHANISTIC EVIDENCE (How, not just that)")
print(f"{'─'*80}")

print(f"\n9.1 Features are genuinely nonlinear in z:")
print(f"  Linear probe R² ~ 0.28, nonlinear probe R² ~ 0.75 (gap = 0.47)")

print(f"\n9.2 NL component works through magnitude modulation (~82%), not ReLU gating (~18%):")
print(f"  Even with NO ReLU flips, ~22% of samples have prediction changes")
print(f"  NL component shifts logits +2.7 for AND=1, -0.7 for AND=0 (correct direction)")

print(f"\n9.3 When predictions disagree, true z is correct 96% for AND=1:")
print(f"  NL catches ~23k AND=1 that linear misses; reverse is only ~400")

print(f"\n9.4 Magnitude changes correlate with AND targets > individual features:")
print(f"  Key neurons show AND corr > feat corr → NOT 'linearize then compute'")

print(f"\n9.5 NL component adds info beyond individual features:")
print(f"  Adding z_nl to (x_i, x_j) improves AND F1 by ~+0.10")
print(f"  But z_nl does NOT linearly encode x_i*x_j (R²~0.02 vs z_lin R²~0.12)")

print(f"\n9.6 All findings robust across 10 seeds (9/10 show correct shift direction)")

print(f"\n{'═'*80}")
print("CONCLUSION")
print(f"{'═'*80}")
print("""
The encoder embeds a small co-activation signal in z's nonlinear component.
The compute head reads this signal directly through magnitude modulation of
already-firing neurons — it does NOT first linearize z to recover individual
features (the magnitude changes correlate with AND, not features).

However, the NL component isn't a stored precomputation of AND either — it
doesn't linearly encode x_i*x_j. It provides a context-dependent hint: a
cooperative signal that, combined with the linear component's neuron gating
pattern, lets the compute head distinguish co-activation from individual
activation.

This is genuine Level 2-3 Nanda evidence with mechanistic backing:
nonlinear features are not just present, they are actively read by
downstream computation in a way that cannot be reduced to "linearize first."
""")

## Discussion

### The Nanda hierarchy: how far do we get?

| Level | Evidence | Our result |
|-------|---------|------------|
| 1 | Nonlinear encoding exists | Yes — 0.5-50% NL fraction depending on seed/objective |
| 2 | Linearizing hurts computation | Yes — F1 drops ~0.16, robust across seeds |
| 3 | Effect is task-specific | Yes — AND >> product ≈ sum |
| 4 | No other explanation | **Partially** — mech interp shows it's NOT "linearize then compute" |

Level 4 is the hardest. Our Part 9 analysis shows the compute head is not first recovering linear features from z — the magnitude changes from z's nonlinear component correlate with AND targets, not individual features. But we also find the NL component doesn't linearly encode interactions (x_i·x_j) — so the mechanism is subtler than "encoder precomputes AND."

### The cooperative mechanism

The picture that emerges:
1. **Linear component of z** (99.5% of variance): determines which compute-head neurons fire and their rough magnitudes. This is the Hänni et al. regime — linear features in superposition.
2. **Nonlinear component of z** (0.5% of variance): provides small context-dependent corrections that modulate firing magnitudes. These corrections are pair-specific (different direction for different co-active pairs) and carry co-activation information that goes beyond individual features.
3. **Compute head**: reads BOTH components cooperatively. The linear part sets up the neuron activation pattern; the nonlinear part adjusts magnitudes to push the AND logit in the right direction. This happens mainly through magnitude changes (~82%), not ReLU gating (~18%).

### What this means for the SAE linearity assumption

SAEs assume features are linear directions in activation space. Our findings suggest:
- Features are **mostly** linear in z (R² ≈ 0.28 from 3 dims), but a nonlinear probe recovers much more (R² ≈ 0.75)
- The nonlinear component is small but **disproportionately important for computation** — SAEs would miss it
- The downstream compute head does NOT first linearize the representation — it uses the nonlinear structure directly
- An SAE applied to z would capture the linear part but miss the co-activation signal that carries ~30% of AND performance

### Connection to LLMs

In real transformers:
- MLP layers perform computation in superposition (Hänni et al.)
- If MLP hidden states carry nonlinear co-activation signals (analogous to our z_nonlinear), SAEs would miss them
- The "feature splitting" phenomenon in SAEs might partly reflect attempts to linearize genuinely nonlinear structure
- Our "magnitude modulation" mechanism could operate in transformer residual streams: small nonlinear corrections that don't change which neurons fire but shift their outputs in task-relevant ways

### Limitations

1. **Toy scale**: n=8 features, m=3 bottleneck. Real models have millions of features.
2. **Known features**: We define the features; in LLMs they're emergent. The "nonlinear features" question is harder when you don't know the ground truth.
3. **Single bottleneck**: Real transformers have many layers of computation, not one shared bottleneck.
4. **Compute head has reconstruction access**: Through the shared encoder, the compute head could potentially learn to undo nonlinearity. Our mech interp evidence argues against this, but the architecture doesn't rule it out.

### Future directions
1. **Single-layer compute head**: Remove the compute head's ability to linearize by using a single Linear(m→k) layer. If nonlinear encoding still helps, the AND info MUST be in z.
2. **Circuit analysis**: Fully reverse-engineer the encoder's internal computation to understand HOW it creates the co-activation signal.
3. **Onion + computation**: Do onion features support the cooperative mechanism differently?
4. **Comparison to Hänni et al. construction**: Does our learned model match their theoretical linear construction, or find a genuinely different (nonlinear) solution?